In [ ]:
import importlib.util
import json
import os
import re
import stat
import subprocess
import sys
import time
import urllib.request
from pathlib import Path


WORKING_DIR = Path("/kaggle/working")
APP_PATH = WORKING_DIR / "app.py"
SERVER_LOG_PATH = WORKING_DIR / "uvicorn.log"
TUNNEL_LOG_PATH = WORKING_DIR / "cloudflared.log"
CLOUDFLARED_PATH = WORKING_DIR / "cloudflared"

LOCAL_SERVER_URL = "http://127.0.0.1:8000"
HEALTH_URL = f"{LOCAL_SERVER_URL}/"


def install_missing_packages():
    required_modules = {
        "faster_whisper": "faster-whisper",
        "pyannote.audio": "pyannote.audio",
        "fastapi": "fastapi",
        "uvicorn": "uvicorn",
        "multipart": "python-multipart",
        "transformers": "transformers>=4.37.0",
        "accelerate": "accelerate",
        "safetensors": "safetensors",
        "requests": "requests",
    }

    missing_packages = []

    for module_name, package_name in required_modules.items():
        try:
            installed = (
                importlib.util.find_spec(module_name)
                is not None
            )
        except ModuleNotFoundError:
            installed = False

        if not installed:
            missing_packages.append(package_name)

    if missing_packages:
        print(
            "Installing:",
            ", ".join(missing_packages),
            flush=True,
        )

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *missing_packages,
        ])


install_missing_packages()


from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret(
    "HF_TOKEN"
)

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN is missing. Add and enable "
        "HF_TOKEN in Kaggle Secrets."
    )

os.environ["HF_TOKEN"] = hf_token


app_code = r'''
from fastapi import (
    FastAPI,
    UploadFile,
    File,
    Form,
    HTTPException,
)
from faster_whisper import WhisperModel
from pyannote.audio import Pipeline
from pydantic import BaseModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)
from typing import Any

import json
import os
import re
import tempfile
import threading
import torch


app = FastAPI(
    title="Call Intelligence API",
    version="1.0.0",
)


# ============================================================
# GPU setup
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select GPU T4 x2 "
        "in Kaggle settings."
    )

gpu_count = torch.cuda.device_count()

if gpu_count < 2:
    raise RuntimeError(
        f"Expected two T4 GPUs, but only "
        f"{gpu_count} GPU was detected."
    )

for gpu_index in range(gpu_count):
    print(
        f"GPU {gpu_index}: "
        f"{torch.cuda.get_device_name(gpu_index)}, "
        f"capability="
        f"{torch.cuda.get_device_capability(gpu_index)}",
        flush=True,
    )


# Whisper and Pyannote run on GPU 0.
whisper_device = "cuda"
pyannote_device = "cuda:0"

# Qwen runs on GPU 1.
qwen_device = "cuda:1"


# ============================================================
# Whisper model
# ============================================================

print(
    f"Loading Faster Whisper on {whisper_device}",
    flush=True,
)

whisper_model = WhisperModel(
    "small.en",
    device=whisper_device,
    compute_type="float16",
)


# ============================================================
# Pyannote model
# ============================================================

hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN environment variable is missing."
    )

print(
    f"Loading Pyannote on {pyannote_device}",
    flush=True,
)

diarization_model = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=hf_token,
)

diarization_model.to(
    torch.device(pyannote_device)
)


# ============================================================
# Qwen model
# ============================================================

qwen_model_id = (
    "Qwen/Qwen2.5-1.5B-Instruct"
)

qwen_tokenizer = None
qwen_model = None

qwen_load_lock = threading.Lock()
qwen_inference_lock = threading.Lock()


def get_qwen_model():
    global qwen_tokenizer
    global qwen_model

    if (
        qwen_tokenizer is not None
        and qwen_model is not None
    ):
        return qwen_tokenizer, qwen_model

    with qwen_load_lock:
        if (
            qwen_tokenizer is None
            or qwen_model is None
        ):
            print(
                f"Loading {qwen_model_id} "
                f"on {qwen_device}",
                flush=True,
            )

            qwen_tokenizer = (
                AutoTokenizer.from_pretrained(
                    qwen_model_id
                )
            )

            qwen_model = (
                AutoModelForCausalLM
                .from_pretrained(
                    qwen_model_id,
                    dtype=torch.float16,
                    low_cpu_mem_usage=True,
                )
                .to(qwen_device)
            )

            qwen_model.eval()

            print(
                "Qwen loaded successfully",
                flush=True,
            )

    return qwen_tokenizer, qwen_model


# ============================================================
# Call-analysis request
# ============================================================

class CallAnalysisRequest(BaseModel):
    systemPrompt: str
    input: Any
    maxNewTokens: int = 600


# ============================================================
# Qwen generation
# ============================================================

def clean_generated_json(generated_text):
    cleaned_text = generated_text.strip()

    cleaned_text = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned_text,
        flags=re.IGNORECASE,
    )

    cleaned_text = re.sub(
        r"\s*```$",
        "",
        cleaned_text,
    )

    return cleaned_text.strip()


def generate_qwen_response(
    system_prompt,
    user_input,
    max_new_tokens,
):
    if not system_prompt.strip():
        raise HTTPException(
            status_code=400,
            detail=(
                "systemPrompt must not be empty"
            ),
        )

    if user_input is None:
        raise HTTPException(
            status_code=400,
            detail="input must not be null",
        )

    if (
        max_new_tokens < 1
        or max_new_tokens > 1200
    ):
        raise HTTPException(
            status_code=400,
            detail=(
                "maxNewTokens must be "
                "between 1 and 1200"
            ),
        )

    user_prompt = (
        user_input
        if isinstance(user_input, str)
        else json.dumps(
            user_input,
            ensure_ascii=False,
        )
    )

    tokenizer, model = get_qwen_model()

    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    model_inputs = (
        tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
        .to(qwen_device)
    )

    with (
        qwen_inference_lock,
        torch.inference_mode(),
    ):
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            repetition_penalty=1.05,
            pad_token_id=(
                tokenizer.eos_token_id
            ),
        )

    input_length = (
        model_inputs["input_ids"].shape[-1]
    )

    generated_tokens = generated_ids[
        :,
        input_length:
    ]

    generated_text = tokenizer.decode(
        generated_tokens[0],
        skip_special_tokens=True,
    ).strip()

    cleaned_text = clean_generated_json(
        generated_text
    )

    try:
        parsed_json = json.loads(
            cleaned_text
        )
    except json.JSONDecodeError:
        parsed_json = None

    return {
        "provider": "qwen",
        "model": qwen_model_id,
        "device": qwen_device,
        "response": cleaned_text,
        "json": parsed_json,
    }


# ============================================================
# Health endpoint
# ============================================================

@app.get("/")
def health():
    return {
        "status": "ok",
        "gpus": [
            {
                "index": index,
                "name": (
                    torch.cuda.get_device_name(
                        index
                    )
                ),
                "capability": list(
                    torch.cuda
                    .get_device_capability(
                        index
                    )
                ),
            }
            for index in range(
                torch.cuda.device_count()
            )
        ],
        "whisper": {
            "provider": "faster-whisper",
            "model": "small.en",
            "device": whisper_device,
        },
        "pyannote": {
            "provider": "pyannote",
            "model": (
                "speaker-diarization-community-1"
            ),
            "device": pyannote_device,
        },
        "qwen": {
            "provider": "qwen",
            "model": qwen_model_id,
            "device": qwen_device,
            "loaded": qwen_model is not None,
        },
    }


# ============================================================
# Transcribe endpoint
# ============================================================

@app.post("/transcribe")
async def transcribe(
    file: UploadFile = File(...),
    needDiarization: bool = Form(False),
):
    suffix = (
        os.path.splitext(
            file.filename or ""
        )[1]
        or ".wav"
    )

    temporary_path = None

    try:
        with tempfile.NamedTemporaryFile(
            delete=False,
            suffix=suffix,
        ) as temporary_file:
            temporary_file.write(
                await file.read()
            )

            temporary_path = (
                temporary_file.name
            )

        segments_iterator, info = (
            whisper_model.transcribe(
                temporary_path,
                language="en",
                word_timestamps=True,
                vad_filter=True,
                beam_size=5,
            )
        )

        transcription_segments = []

        for segment in segments_iterator:
            words = []

            if segment.words:
                for word in segment.words:
                    words.append({
                        "word": word.word,
                        "start": word.start,
                        "end": word.end,
                        "probability": (
                            word.probability
                        ),
                    })

            transcription_segments.append({
                "id": segment.id,
                "seek": segment.seek,
                "start": segment.start,
                "end": segment.end,
                "text": segment.text,
                "temperature": (
                    segment.temperature
                ),
                "avg_logprob": (
                    segment.avg_logprob
                ),
                "compression_ratio": (
                    segment.compression_ratio
                ),
                "no_speech_prob": (
                    segment.no_speech_prob
                ),
                "words": words,
            })

        diarization_segments = []
        exclusive_segments = []

        if needDiarization:
            diarization_output = (
                diarization_model(
                    temporary_path
                )
            )

            for (
                turn,
                speaker,
            ) in (
                diarization_output
                .speaker_diarization
            ):
                diarization_segments.append({
                    "start": turn.start,
                    "end": turn.end,
                    "speaker": speaker,
                })

            for (
                turn,
                speaker,
            ) in (
                diarization_output
                .exclusive_speaker_diarization
            ):
                exclusive_segments.append({
                    "start": turn.start,
                    "end": turn.end,
                    "speaker": speaker,
                })

        return {
            "whisper": {
                "provider": (
                    "faster-whisper"
                ),
                "model": "small.en",
                "language": info.language,
                "language_probability": (
                    info.language_probability
                ),
                "duration": info.duration,
                "duration_after_vad": (
                    info.duration_after_vad
                ),
                "segments": (
                    transcription_segments
                ),
            },
            "pyannote": {
                "provider": "pyannote",
                "model": (
                    "speaker-diarization-"
                    "community-1"
                ),
                "requested": needDiarization,
                "performed": needDiarization,
                "diarization": (
                    diarization_segments
                ),
                "exclusive_diarization": (
                    exclusive_segments
                ),
            },
        }

    except Exception as error:
        print(
            "Transcription error:",
            repr(error),
            flush=True,
        )

        raise HTTPException(
            status_code=500,
            detail=str(error),
        )

    finally:
        if (
            temporary_path
            and os.path.exists(
                temporary_path
            )
        ):
            os.unlink(temporary_path)


# ============================================================
# Call-analysis endpoint
# ============================================================

@app.post("/callAnalysis")
def call_analysis(
    request: CallAnalysisRequest,
):
    try:
        return generate_qwen_response(
            system_prompt=(
                request.systemPrompt
            ),
            user_input=request.input,
            max_new_tokens=(
                request.maxNewTokens
            ),
        )

    except HTTPException:
        raise

    except Exception as error:
        print(
            "Call analysis error:",
            repr(error),
            flush=True,
        )

        raise HTTPException(
            status_code=500,
            detail=str(error),
        )
'''


APP_PATH.write_text(
    app_code,
    encoding="utf-8",
)

print(
    "Created",
    APP_PATH,
    flush=True,
)


# Stop the previous server on port 8000.

subprocess.run(
    [
        "fuser",
        "-k",
        "8000/tcp",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=False,
)

time.sleep(1)


# Start Uvicorn.

server_log = open(
    SERVER_LOG_PATH,
    "w",
    encoding="utf-8",
)

server_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "app:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
        "--workers",
        "1",
    ],
    cwd=WORKING_DIR,
    stdout=server_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

print(
    "Loading Whisper and Pyannote...",
    flush=True,
)


# Wait for the server to become healthy.

health_response = None

for attempt in range(120):
    if server_process.poll() is not None:
        server_log.flush()

        raise RuntimeError(
            "Uvicorn stopped during startup:\n"
            + SERVER_LOG_PATH.read_text(
                encoding="utf-8"
            )
        )

    try:
        import requests

        response = requests.get(
            HEALTH_URL,
            timeout=5,
        )

        if response.ok:
            health_response = response.json()
            break

    except requests.RequestException:
        pass

    if attempt % 6 == 0:
        print(
            f"Still loading models "
            f"({attempt * 5}s)...",
            flush=True,
        )

    time.sleep(5)


if health_response is None:
    server_process.terminate()
    server_log.flush()

    raise RuntimeError(
        "Server did not become healthy:\n"
        + SERVER_LOG_PATH.read_text(
            encoding="utf-8"
        )
    )


print(
    "Health:",
    json.dumps(
        health_response,
        indent=2,
    ),
    flush=True,
)


# Download Cloudflared.

if not CLOUDFLARED_PATH.exists():
    print(
        "Downloading cloudflared...",
        flush=True,
    )

    urllib.request.urlretrieve(
        (
            "https://github.com/cloudflare/"
            "cloudflared/releases/latest/"
            "download/"
            "cloudflared-linux-amd64"
        ),
        CLOUDFLARED_PATH,
    )

    CLOUDFLARED_PATH.chmod(
        CLOUDFLARED_PATH.stat().st_mode
        | stat.S_IXUSR
        | stat.S_IXGRP
        | stat.S_IXOTH
    )


# Start Cloudflare tunnel.

tunnel_log = open(
    TUNNEL_LOG_PATH,
    "w",
    encoding="utf-8",
)

tunnel_process = subprocess.Popen(
    [
        str(CLOUDFLARED_PATH),
        "tunnel",
        "--url",
        LOCAL_SERVER_URL,
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)


# Wait for the public URL.

public_url = None

url_pattern = re.compile(
    r"https://[a-zA-Z0-9.-]+"
    r"\.trycloudflare\.com"
)

for _ in range(60):
    if tunnel_process.poll() is not None:
        tunnel_log.flush()

        raise RuntimeError(
            "Cloudflare tunnel stopped:\n"
            + TUNNEL_LOG_PATH.read_text(
                encoding="utf-8"
            )
        )

    tunnel_log.flush()

    log_text = (
        TUNNEL_LOG_PATH.read_text(
            encoding="utf-8"
        )
    )

    match = url_pattern.search(log_text)

    if match:
        public_url = match.group(0)
        break

    time.sleep(1)


if not public_url:
    raise RuntimeError(
        "Cloudflare URL was not created:\n"
        + TUNNEL_LOG_PATH.read_text(
            encoding="utf-8"
        )
    )


print(
    "\nPUBLIC URL:",
    public_url,
    flush=True,
)

print(
    "HEALTH URL:",
    public_url + "/",
    flush=True,
)

print(
    "TRANSCRIBE URL:",
    public_url + "/transcribe",
    flush=True,
)

print(
    "CALL ANALYSIS URL:",
    public_url + "/callAnalysis",
    flush=True,
)